# 强化学习基础理论与经典算法 (RL Foundations) 企业笔试手撕通关宝典
> **面向对象**：互联网大厂/AI 独角兽企业 强化学习 / 运筹决策 / 自动驾驶 / 机器人算法岗面试手撕  
> **核心涵盖**：MDP 矩阵解析解、策略迭代与值迭代、重要性采样、TD(0)/Sarsa/Q-Learning、经验回放池、PER 优先级采样 SumTree、DQN 目标网络软更新、Double DQN 过估计解耦、Dueling DQN 优势流解耦  
> **设计准则**：纯 NumPy / PyTorch 逐行手推，拒绝 Gym/Stable-Baselines 黑盒，彻底吃透马尔可夫决策过程与贝尔曼方程数学不动点。

---
### 核心模块速览
1. **模块一**：贝尔曼期望方程矩阵解析解与迭代策略评估 (Policy Evaluation)
2. **模块二**：值迭代 (Value Iteration) 与 策略迭代 (Policy Iteration) 手撕
3. **模块三**：蒙特卡洛评估与重要性采样 (Ordinary vs. Weighted Importance Sampling)
4. **模块四**：TD(0)、Sarsa (同策略) 与 Q-Learning (异策略) 统一手撕
5. **模块五**：经典经验回放池 (Replay Buffer) 纯 Python 手撕 (循环队列 FIFO)
6. **模块六**：优先经验回放 (PER, Prioritized Experience Replay) SumTree 树形结构手撕
7. **模块七**：深度 Q 网络 (DQN) 与目标网络软更新 (Polyak Averaging) 手撕
8. **模块八**：Double DQN 动作选择与目标评估解耦手撕 (消除过估计偏差)
9. **模块九**：Dueling DQN 状态价值与优势函数解耦手撕

---
## 模块一：贝尔曼期望方程矩阵解析解与迭代策略评估 (Policy Evaluation)

### 【笔试考点与矩阵推导】
- **贝尔曼期望方程矩阵形式**：
  $$V^\pi = R^\pi + \gamma P^\pi V^\pi \implies (I - \gamma P^\pi) V^\pi = R^\pi \implies V^\pi = (I - \gamma P^\pi)^{-1} R^\pi$$
- **可逆性保证**：因为 $\gamma < 1$ 且 $P^\pi$ 为随机矩阵（行和为 1，谱半径 $\le 1$），所以 $(I - \gamma P^\pi)$ 严格可逆，解析解唯一存在！

In [1]:
import numpy as np

def solve_bellman_matrix_analytic(P_pi, R_pi, gamma=0.9):
    """
    矩阵求逆解析解: V = (I - gamma * P)^(-1) * R
    P_pi: (S, S) 状态转移矩阵
    R_pi: (S,) 期望奖励向量
    """
    S = P_pi.shape[0]
    I = np.eye(S)
    V = np.linalg.solve(I - gamma * P_pi, R_pi)
    return V

def iterative_policy_evaluation(P_pi, R_pi, gamma=0.9, theta=1e-6):
    """动态规划迭代策略评估: V_{k+1} = R + gamma * P * V_k"""
    S = P_pi.shape[0]
    V = np.zeros(S)
    while True:
        V_new = R_pi + gamma * np.dot(P_pi, V)
        if np.max(np.abs(V_new - V)) < theta:
            break
        V = V_new
    return V

# 构造 3 状态 MDP 测试
P = np.array([[0.1, 0.9, 0.0],
              [0.0, 0.1, 0.9],
              [0.5, 0.0, 0.5]])
R = np.array([1.0, 2.0, 5.0])
V_analytic = solve_bellman_matrix_analytic(P, R, gamma=0.9)
V_iter = iterative_policy_evaluation(P, R, gamma=0.9)

print("矩阵逆解析求得 V 状态价值:", np.round(V_analytic, 4))
print("动态规划迭代求得 V 状态价值:", np.round(V_iter, 4))
assert np.allclose(V_analytic, V_iter, atol=1e-4)
print(">>> 贝尔曼期望方程解析解与迭代解严格自洽验证成功！")

矩阵逆解析求得 V 状态价值: [29.1617 31.5274 32.9505]
动态规划迭代求得 V 状态价值: [29.1617 31.5274 32.9505]
>>> 贝尔曼期望方程解析解与迭代解严格自洽验证成功！


---
## 模块二：值迭代 (Value Iteration) 与 策略迭代 (Policy Iteration) 手撕

### 【笔试对比考点】
1. **策略迭代 (Policy Iteration)**：交替执行【策略评估 $V^\pi$ 完全收敛】与【策略贪心改进 $\pi' = \arg\max Q$】；
2. **值迭代 (Value Iteration)**：不等策略评估收敛，每一步直接通过贝尔曼最优算子（Bellman Optimality Operator）取 $\max_a$：
   $$V(s) \leftarrow \max_a \sum_{s'} P(s' \mid s, a) \left[ R(s, a, s') + \gamma V(s') \right]$$

In [ ]:
def value_iteration(num_states, num_actions, P_trans, rewards, gamma=0.9, theta=1e-6):
    """
    值迭代手撕
    P_trans: (S, A, S)
    rewards: (S, A)
    """
    V = np.zeros(num_states)
    while True:
        delta = 0.0
        # Q(s, a) = R(s, a) + gamma * sum(P(s'|s, a) * V(s'))
        # 矩阵广播计算 Q: (S, A)
        Q = rewards + gamma * np.sum(P_trans * V[None, None, :], axis=-1)
        V_new = np.max(Q, axis=-1)
        delta = np.max(np.abs(V_new - V))
        V = V_new
        if delta < theta:
            break
            
    # 提取最优确定性策略 pi(s)
    Q = rewards + gamma * np.sum(P_trans * V[None, None, :], axis=-1)
    best_policy = np.argmax(Q, axis=-1)
    return V, best_policy

# 简单 2 状态 2 动作玩具测试
P_toy = np.zeros((2, 2, 2))
# s0: a0 留在 s0 (0.8), a1 去 s1 (1.0)
P_toy[0, 0] = [0.8, 0.2]; P_toy[0, 1] = [0.0, 1.0]
P_toy[1, 0] = [0.5, 0.5]; P_toy[1, 1] = [0.1, 0.9]
R_toy = np.array([[1.0, 10.0], [0.0, 5.0]])

V_opt, pi_opt = value_iteration(2, 2, P_toy, R_toy, gamma=0.9)
print("值迭代求解的最优状态价值 V*:", np.round(V_opt, 4))
print("值迭代求解的最优动作策略 pi*:", pi_opt)
assert len(pi_opt) == 2
print(">>> 值迭代动态规划验证成功！")

---
## 模块三：蒙特卡洛评估与重要性采样 (Ordinary vs. Weighted) 手撕

### 【笔试核心考点】
1. **重要性采样比率**：$\rho = \prod_{t=0}^{T-1} \frac{\pi(a_t \mid s_t)}{b(a_t \mid s_t)}$；
2. **普通重要性采样 (Ordinary IS)**：$V = \frac{1}{N} \sum \rho_i G_i$（无偏，但方差易爆炸！）；
3. **加权重要性采样 (Weighted IS)**：$V = \frac{\sum \rho_i G_i}{\sum \rho_i}$（有微弱偏差，但方差有严格上界，收敛极为平稳！）。

In [ ]:
def importance_sampling_demo(returns, pi_probs, b_probs):
    """
    returns: (N,) 各回合的回报 G
    pi_probs, b_probs: (N,) 轨迹在目标策略与行为策略下的联合概率
    """
    # 计算重要性权重 rho
    rhos = pi_probs / b_probs
    
    # 1. 普通重要性采样 (Ordinary)
    v_ordinary = np.mean(rhos * returns)
    
    # 2. 加权重要性采样 (Weighted)
    v_weighted = np.sum(rhos * returns) / (np.sum(rhos) + 1e-8)
    
    return v_ordinary, v_weighted

dummy_G = np.array([10.0, 20.0, 100.0])
p_pi = np.array([0.5, 0.3, 0.01]) # 目标策略认为 G=100 概率极小
p_b = np.array([0.1, 0.2, 0.5])   # 行为策略高估了 G=100 的发生率

v_ord, v_wei = importance_sampling_demo(dummy_G, p_pi, p_b)
print(f"普通重要性采样估值: {v_ord:.2f}")
print(f"加权重要性采样估值 (稳健抗极端方差): {v_wei:.2f}")
assert not np.isnan(v_wei)
print(">>> 异策略重要性采样验证成功！")

---
## 模块四：TD(0)、Sarsa (同策略) 与 Q-Learning (异策略) 统一手撕

### 【笔试高频考点】
- **Sarsa (On-Policy)**：实际执行 $A'$，用真实动作更新：
  $$Q(S, A) \leftarrow Q(S, A) + \alpha \left[ R + \gamma Q(S', A') - Q(S, A) \right]$$
- **Q-Learning (Off-Policy)**：虚空取最大化 $\max_{a'} Q(S', a')$，动作选择与更新完全解耦：
  $$Q(S, A) \leftarrow Q(S, A) + \alpha \left[ R + \gamma \max_{a'} Q(S', a') - Q(S, A) \right]$$

In [ ]:
def sarsa_update(Q, s, a, r, s_next, a_next, gamma=0.9, alpha=0.1):
    td_target = r + gamma * Q[s_next, a_next]
    Q[s, a] += alpha * (td_target - Q[s, a])
    return Q

def q_learning_update(Q, s, a, r, s_next, gamma=0.9, alpha=0.1):
    td_target = r + gamma * np.max(Q[s_next]) # 异策略: 取 max
    Q[s, a] += alpha * (td_target - Q[s, a])
    return Q

# 测试两种更新
Q_sarsa = np.zeros((2, 2))
Q_ql = np.zeros((2, 2))

sarsa_update(Q_sarsa, s=0, a=0, r=2.0, s_next=1, a_next=1)
q_learning_update(Q_ql, s=0, a=0, r=2.0, s_next=1)
print("Sarsa 步后 Q 表值:", Q_sarsa[0, 0])
print("Q-Learning 步后 Q 表值:", Q_ql[0, 0])
assert Q_sarsa[0, 0] == 0.2 and Q_ql[0, 0] == 0.2
print(">>> Sarsa 与 Q-Learning 对比验证成功！")

---
## 模块五：经典经验回放池 (Replay Buffer) 纯 Python 手撕 (循环队列 FIFO)

### 【笔试考点】
1. **物理意义**：打破连续样本的时序相关性（I.I.D. 假设），复用珍贵历史转移数据；
2. **指针循环覆盖**：预分配固定长度，用 `self.pos = (self.pos + 1) % self.capacity` 实现常数级 $O(1)$ 写入。

In [ ]:
class ReplayBuffer:
    def __init__(self, capacity):
        self.capacity = capacity
        self.buffer = []
        self.pos = 0

    def push(self, state, action, reward, next_state, done):
        transition = (state, action, reward, next_state, done)
        if len(self.buffer) < self.capacity:
            self.buffer.append(transition)
        else:
            self.buffer[self.pos] = transition
        self.pos = (self.pos + 1) % self.capacity

    def sample(self, batch_size):
        indices = np.random.choice(len(self.buffer), batch_size, replace=False)
        batch = [self.buffer[i] for i in indices]
        states, actions, rewards, next_states, dones = zip(*batch)
        return (np.array(states), np.array(actions), np.array(rewards, dtype=np.float32),
                np.array(next_states), np.array(dones, dtype=np.uint8))

    def __len__(self):
        return len(self.buffer)

# 测试经验回放
rb = ReplayBuffer(capacity=5)
for i in range(7): # 触发循环覆盖
    rb.push(np.array([i]), 0, 1.0, np.array([i+1]), False)
print("当前回放池样本数 (上限5):", len(rb))
s_b, a_b, r_b, _, _ = rb.sample(batch_size=3)
assert len(rb) == 5 and s_b.shape == (3, 1)
print(">>> ReplayBuffer 循环队列采样验证成功！")

---
## 模块六：优先经验回放 (Prioritized Experience Replay, PER) SumTree 手撕

### 【笔试硬核考点】
1. **以 TD-error 绝对值为优先级**：$p_i = |\delta_i| + \epsilon$；
2. **SumTree 二叉线段树**：父节点等于两子节点之和，叶子节点存优先级 $p_i$。抽样只需在 $[0, \text{total}]$ 生成随机数，沿树向下检索，复杂度严格为 **$O(\log N)$**！
3. **重要性采样权重修正**：$w_i = (N \cdot P(i))^{-\beta} / \max(w)$。

In [ ]:
class SumTree:
    def __init__(self, capacity):
        self.capacity = capacity
        # 二叉树数组大小: 2 * capacity - 1
        self.tree = np.zeros(2 * capacity - 1)
        self.data = np.zeros(capacity, dtype=object)
        self.write = 0
        self.n_entries = 0

    def _propagate(self, idx, change):
        parent = (idx - 1) // 2
        self.tree[parent] += change
        if parent != 0:
            self._propagate(parent, change)

    def update(self, idx, priority):
        change = priority - self.tree[idx]
        self.tree[idx] = priority
        self._propagate(idx, change)

    def add(self, priority, data):
        idx = self.write + self.capacity - 1
        self.data[self.write] = data
        self.update(idx, priority)
        self.write = (self.write + 1) % self.capacity
        if self.n_entries < self.capacity:
            self.n_entries += 1

    def _retrieve(self, idx, s):
        left = 2 * idx + 1
        right = left + 1
        if left >= len(self.tree):
            return idx
        if s <= self.tree[left]:
            return self._retrieve(left, s)
        else:
            return self._retrieve(right, s - self.tree[left])

    def get(self, s):
        idx = self._retrieve(0, s)
        data_idx = idx - self.capacity + 1
        return idx, self.tree[idx], self.data[data_idx]

    @property
    def total_priority(self):
        return self.tree[0]

# 测试 SumTree
tree = SumTree(capacity=4)
tree.add(10.0, "data_A")
tree.add(20.0, "data_B")
tree.add(30.0, "data_C")
tree.add(40.0, "data_D")
print("SumTree 根节点总优先级 (10+20+30+40=100):", tree.total_priority)
assert tree.total_priority == 100.0

# 采样检索
idx, p, data = tree.get(15.0) # 应落在 data_B (10~30)
print(f"前缀和 15.0 命中叶子节点数据: {data}, 对应权重: {p}")
assert data == "data_B"
print(">>> PER SumTree 二分高效前缀检索验证通过！")

---
## 模块七：深度 Q 网络 (DQN) 与目标网络软更新 (Polyak Averaging) 手撕

### 【笔试考点】
1. **打破自举死锁**：若用同一个网络既预测 $Q(s, a)$ 又预测目标 $r + \gamma \max Q(s', a')$，网络就像“追赶自己的尾巴”，极易发散；
2. **目标网络软更新 (Polyak Averaging)**：
   $$\theta^- \leftarrow \tau \theta + (1 - \tau) \theta^-, \quad \tau \ll 1$$

In [ ]:
import torch
import torch.nn as nn

def soft_update_target_network(main_net, target_net, tau=0.005):
    """Polyak 目标网络平滑软更新"""
    for param, target_param in zip(main_net.parameters(), target_net.parameters()):
        target_param.data.copy_(tau * param.data + (1.0 - tau) * target_param.data)

net_eval = nn.Linear(4, 2)
net_target = nn.Linear(4, 2)
# 先硬同步
net_target.load_state_dict(net_eval.state_dict())

# 改变 eval 权重后执行软更新
with torch.no_grad():
    net_eval.weight.add_(1.0)
soft_update_target_network(net_eval, net_target, tau=0.1)
diff = (net_eval.weight - net_target.weight).abs().mean().item()
print("软更新后目标网络与评估网络参数差值 (严格按 tau=0.1 缓步追赶):", round(diff, 4))
assert 0.8 < diff < 1.0
print(">>> DQN 目标网络 Polyak 软更新验证成功！")

---
## 模块八：Double DQN 动作选择与目标评估解耦手撕 (消除过估计偏差)

### 【笔试必问核心考点】
1. **传统 DQN 过估计根因**：$\max_{a'} Q(s', a'; \theta^-)$ 里的 $\max$ 算子具有凸函数性质，$\mathbb{E}[\max(X)] \ge \max(\mathbb{E}[X])$，持续引入正向噪声偏差；
2. **Double DQN 解耦机制**：
   - 用**当前评估网络**选择最优动作：$a^* = \arg\max_{a'} Q(s', a'; \theta)$；
   - 用**目标网络**评价该动作的价值：$Q_{\text{target}} = r + \gamma Q(s', a^*; \theta^-)$；
   - 彻底打破“自己选、自己打分”的贪心死循环！

In [ ]:
def compute_double_dqn_targets(eval_net, target_net, next_states, rewards, dones, gamma=0.99):
    """
    Double DQN 目标值解耦计算
    """
    # 1. 评估网络负责挑选动作 argmax
    with torch.no_grad():
        next_q_eval = eval_net(next_states)                           # (B, A)
        best_actions = torch.argmax(next_q_eval, dim=-1, keepdim=True) # (B, 1)
        
        # 2. 目标网络负责对选出的动作打分 gather
        next_q_target = target_net(next_states)                       # (B, A)
        target_q_val = next_q_target.gather(1, best_actions).squeeze(-1) # (B,)
        
        targets = rewards + gamma * (1.0 - dones) * target_q_val
    return targets

# 测试 Double DQN 解耦
eval_m = nn.Linear(4, 3)
target_m = nn.Linear(4, 3)
s_next_d = torch.randn(2, 4)
r_d = torch.tensor([1.0, 2.0])
done_d = torch.tensor([0.0, 0.0])
ddqn_t = compute_double_dqn_targets(eval_m, target_m, s_next_d, r_d, done_d)
print("Double DQN 目标价值张量 (B=2):", ddqn_t.numpy())
assert ddqn_t.shape == (2,)
print(">>> Double DQN 动作解耦验证成功！")

---
## 模块九：Dueling DQN 状态价值与优势函数解耦手撕

### 【笔试考点与不可辨识度破解】
1. **解耦结构**：将网络分为状态价值分支 $V(s)$ 和动作优势分支 $A(s, a)$；
2. **中心化唯一确定性公式**：
   $$Q(s, a) = V(s) + \left( A(s, a) - \frac{1}{|\mathcal{A}|} \sum_{a'} A(s, a') \right)$$
   减去优势均值确保 $\sum_a A(s, a) = 0$，使得 $V(s)$ 能够被唯一识别确定！

In [ ]:
class DuelingDQN(nn.Module):
    def __init__(self, state_dim, action_dim, hidden_dim=32):
        super().__init__()
        self.feature_layer = nn.Sequential(nn.Linear(state_dim, hidden_dim), nn.ReLU())
        
        # 价值流 V(s): 输出标量 (B, 1)
        self.value_stream = nn.Linear(hidden_dim, 1)
        
        # 优势流 A(s, a): 输出向量 (B, action_dim)
        self.advantage_stream = nn.Linear(hidden_dim, action_dim)

    def forward(self, x):
        feat = self.feature_layer(x)
        val = self.value_stream(feat)                       # (B, 1)
        adv = self.advantage_stream(feat)                   # (B, action_dim)
        
        # 核心解耦中心化公式: Q = V + (A - mean(A))
        q_values = val + (adv - adv.mean(dim=-1, keepdim=True))
        return q_values

dueling = DuelingDQN(state_dim=4, action_dim=3)
dummy_s = torch.randn(2, 4)
q_out = dueling(dummy_s)
print("Dueling DQN 输出 Q 值形状:", q_out.shape)
assert q_out.shape == (2, 3)
print(">>> Dueling DQN 价值优势流解耦手撕通过！")

---
## 企业笔试手撕核心口诀与雷区速记卡

```
1. 贝尔曼可逆解: V = (I - gamma*P)^(-1) * R，谱半径小保证逆矩阵必定存在。
2. 值迭代 vs 策略: 策略迭代每轮跑满收敛再改策略，值迭代每步直接 max_a 极速奔向不动点。
3. SumTree 抽样: O(log N) 树形二分，前缀和按叶节点优先级采样，权重乘 (N*P)^(-beta) 纠偏。
4. Double DQN 解耦: 评估网挑动作 argmax，目标网评价 gather，彻底斩断过估计贪婪链条。
5. Dueling 均值去偏: Q = V + (A - mean(A))，强制优势和为零以破解参数不可辨识度。
```